# Understand the remaining cassette detections

**CPU only. Run all four code cells.** This notebook reproduces a completed
analysis using saved predictions from the same 20 development photographs.
It needs no GPU, image download, model download or training.

First it explains which outputs are extra, which are duplicate, and which
missed fronts have incomplete corner annotations. Then it tests one fixed
alternative: retain the reflected view's boxes and use the same equal
corner averaging. It does not search thresholds or change annotations.

Overlap groups are review cues, not proof that an output is background.
The different annotation references describe the same photographs. This
notebook does not replace the pending classification-loss or training-label
experiments, and it does not select an application model.


## 1 Load the fixed implementation

The source is pinned to a published commit and kept separate from output.
If this directory already contains edited source, choose another
`BAOMAI_DETECTION_AUDIT_BASE` location.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "f41ce86d49aec0df7acba0389fd0624e3325fd35"
DRIVER_HASHES = {'analyze_detection_errors_r62.py': 'd9a2c367b24290318b9104cf4d82b1ca72cf1c29500121c9d23976ee936f21eb', 'probe_reflected_anchor_r62.py': '359fc31df23f17e5b99b306b5aa57aaa5319681428165de7033655fc28d02d85'}
BASE = Path(os.environ.get("BAOMAI_DETECTION_AUDIT_BASE", "/content/baomai-detection-audit-r62")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing source preserved; choose a fresh BAOMAI_DETECTION_AUDIT_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError("Changed source preserved; choose a fresh BAOMAI_DETECTION_AUDIT_BASE.")
for name, expected in DRIVER_HASHES.items():
    if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Source identity differs: " + name)
WORK.mkdir(parents=True, exist_ok=True)
print("Published source:", SOURCE_COMMIT)


hint: Using 'master' as the name for the initial branch. This default branch name
hint: will change to "main" in Git 3.0. To configure the initial branch name
hint: to use in all of your new repositories, which will suppress this warning,
hint: call:
hint:
hint: 	git config --global init.defaultBranch <name>
hint:
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint:
hint: 	git branch -m <name>
hint:
hint: Disable this message with "git config set advice.defaultBranchName false"
Initialized empty Git repository in /workspace/scratch/f5388ec04e55/r62-notebook-run/source/.git/
From https://github.com/Siuuuers/dl
 * branch            f41ce86d49aec0df7acba0389fd0624e3325fd35 -> FETCH_HEAD
HEAD is now at f41ce86 Diagnose partial-front errors and test reflected detection boxes
Published source: f41ce86d49aec0df7acba0389fd0624e3325fd35


## 2 Reproduce and partition the current errors

The fixed archive and all its members are checked. The original scores,
associations and predictions must reproduce exactly before new counts
are reported. Duplicate outputs remain a subset of extras. The two other
groups use maximum reference IoU below 0.10 or from 0.10 to below 0.50.

Missing corners do not remove their fronts from detection evaluation.
Saved native outputs already passed confidence 0.25 and ordinary NMS,
so this cache cannot identify every reason a lower-scored or suppressed
proposal is absent.


In [2]:
AUDIT = WORK / "detection-errors"
run([sys.executable, "-B", SOURCE / "scripts/analyze_detection_errors_r62.py",
     "--root", SOURCE, "--output", AUDIT])
audit = json.loads((AUDIT / "summary.json").read_text())
from IPython.display import Markdown, display
display(Markdown((AUDIT / "results.md").read_text()))
print("Review records:", audit["review_case_count"])


{
  "status": "completed",
  "verification": {
    "archive_members_and_hashes": "passed",
    "native_output_reconstruction": "exact",
    "saved_score_and_case_replay": "exact",
    "original_proposed_matrix_rows": 6,
    "per_image_scores_replayed": 240
  },
  "primary_headline": {
    "fronts": 654,
    "outputs": 687,
    "matched": 636,
    "missed": 18,
    "extras": 51,
    "duplicates": 5,
    "eligible_pairs": 583,
    "within5": 569,
    "within10": 583,
    "extra_strata": {
      "duplicate": 5,
      "near_reference_overlap": 36,
      "low_reference_overlap": 10
    },
    "by_available_corners": {
      "0": {
        "fronts": 17,
        "missed": 5
      },
      "1": {
        "fronts": 54,
        "missed": 13
      },
      "2": {
        "fronts": 583,
        "missed": 0
      }
    },
    "miss_cache_stages": {
      "no_eligible_cached_candidate": 18
    }
  },
  "output_sha256": {
    "cases.json": "982e5578f43b922f0723c5d305ccf4b9be2faeb35fd24781c553eee160b5

# Cached detection error audit

All six baseline rows reproduce exactly; no inference or training was performed.

| Reference | Method | Missed | Extras | Duplicate subset | Other IoU 0.10–<0.50 | Other IoU <0.10 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| supplied664 | original | 39 | 56 | 11 | 38 | 7 |
| supplied664 | proposed | 46 | 69 | 8 | 52 | 9 |
| frozen666 | original | 26 | 41 | 11 | 25 | 5 |
| frozen666 | proposed | 34 | 55 | 9 | 39 | 7 |
| r50 | original | 34 | 61 | 5 | 49 | 7 |
| r50 | proposed | 18 | 51 | 5 | 36 | 10 |

## Current method under the reviewed reference

| Locatable reference corners | Fronts | Missed |
| --- | ---: | ---: |
| 0 | 17 | 5 |
| 1 | 54 | 13 |
| 2 | 583 | 0 |

## Interpretation limits

- Overlap strata are geometric review cues, not physical background, label-error or object-identity verdicts.
- The native cache was already filtered at confidence 0.25, ordinary NMS 0.70 and max_det 300, then clipped. It cannot expose discarded native proposals or lower-score responses.
- No eligible cached candidate does not establish that the network produced no candidate. New pre-filter decoding would be needed to separate confidence, native NMS and proposal failures.
- Pipeline stages identify where eligible records disappear within the saved cache only; they do not establish a physical cause.
- Three annotation references describe the same repeatedly inspected 20 development photographs and are not independent test sets.
- Corner availability is reference-defined. A missing corner does not remove its front from detection evaluation.
- Reflected-view complementarity is an annotation-based diagnostic, not an implementable oracle or evidence that merging views improves the result.
- This replay changes no labels, predictions, model, matching rule, confidence cutoff or deployment setting.


## 3 Reproduce the one fixed reflected-box experiment

Keep reflected-view boxes, scores and identities. Associate each with
the original view using mutual best predicted-box IoU at least 0.50;
average the two corresponding corners equally when both pairs are
available. Otherwise retain the reflected record. No reference label
chooses the associations or weights.

The detection confidence stays strictly above 0.25, NMS overlap at 0.70,
reference-matching IoU at least 0.50 and corner confidence at least 0.50.
All three reference versions are reported, including adverse results.


In [3]:
PROBE = WORK / "reflected-anchor"
run([sys.executable, "-B", SOURCE / "scripts/probe_reflected_anchor_r62.py",
     "--root", SOURCE, "--work", PROBE])
probe = json.loads((PROBE / "summary.json").read_text())
rows = ["| Reference | Method | Misses | Extras | Duplicates | Both within 5 pixels | Both within 10 pixels |",
        "| --- | --- | ---: | ---: | ---: | ---: | ---: |"]
for r in probe["matrix"]:
    rows.append(f"| {r['reference']} | {r['method']} | {r['missed']} | {r['extras']} | {r['duplicates']} | {r['within5']}/{r['eligible_pairs']} | {r['within10']}/{r['eligible_pairs']} |")
display(Markdown("\n".join(rows)))
print("This exploratory choice has tradeoffs; no model is promoted.")


supplied664 original missed 39 extras 56 duplicates 11 pairs5 521/584 pairs10 568/584
supplied664 proposed missed 46 extras 69 duplicates 8 pairs5 514/584 pairs10 565/584
supplied664 reflected missed 47 extras 65 duplicates 9 pairs5 507/584 pairs10 562/584
supplied664 reflected_anchor missed 47 extras 65 duplicates 9 pairs5 512/584 pairs10 563/584
frozen666 original missed 26 extras 41 duplicates 11 pairs5 534/584 pairs10 580/584
frozen666 proposed missed 34 extras 55 duplicates 9 pairs5 525/584 pairs10 574/584
frozen666 reflected missed 34 extras 50 duplicates 12 pairs5 518/584 pairs10 572/584
frozen666 reflected_anchor missed 34 extras 50 duplicates 12 pairs5 524/584 pairs10 573/584
r50 original missed 34 extras 61 duplicates 5 pairs5 555/583 pairs10 577/583
r50 proposed missed 18 extras 51 duplicates 5 pairs5 569/583 pairs10 583/583
r50 reflected missed 15 extras 43 duplicates 10 pairs5 562/583 pairs10 583/583
r50 reflected_anchor missed 15 extras 43 duplicates 10 pairs5 569/583 pai

| Reference | Method | Misses | Extras | Duplicates | Both within 5 pixels | Both within 10 pixels |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| supplied664 | original | 39 | 56 | 11 | 521/584 | 568/584 |
| supplied664 | proposed | 46 | 69 | 8 | 514/584 | 565/584 |
| supplied664 | reflected | 47 | 65 | 9 | 507/584 | 562/584 |
| supplied664 | reflected_anchor | 47 | 65 | 9 | 512/584 | 563/584 |
| frozen666 | original | 26 | 41 | 11 | 534/584 | 580/584 |
| frozen666 | proposed | 34 | 55 | 9 | 525/584 | 574/584 |
| frozen666 | reflected | 34 | 50 | 12 | 518/584 | 572/584 |
| frozen666 | reflected_anchor | 34 | 50 | 12 | 524/584 | 573/584 |
| r50 | original | 34 | 61 | 5 | 555/583 | 577/583 |
| r50 | proposed | 18 | 51 | 5 | 569/583 | 583/583 |
| r50 | reflected | 15 | 43 | 10 | 562/583 | 583/583 |
| r50 | reflected_anchor | 15 | 43 | 10 | 569/583 | 583/583 |

## 4 Verify the saved result and download

Recompute both analyses from the fixed input and compare every saved
result. The compact ZIP preserves summary tables, exact review cases,
paired changes, view associations and source identities. The analysis
is already completed; rerunning is optional.


In [4]:
import gzip
from zipfile import ZipFile, ZipInfo, ZIP_DEFLATED
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
from analyze_detection_errors_r62 import analyze as analyze_errors, markdown as audit_markdown
from probe_reflected_anchor_r62 import analyze as analyze_probe, ARCHIVE_PATH

if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError("Source changed after setup")
for name, expected in DRIVER_HASHES.items():
    if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Source identity differs: " + name)
replay_audit, replay_cases = analyze_errors(SOURCE)
if replay_audit != json.loads((AUDIT / "summary.json").read_text()) or replay_cases != json.loads((AUDIT / "cases.json").read_text()):
    raise RuntimeError("Saved error audit differs from replay")
if (AUDIT / "results.md").read_text() != audit_markdown(replay_audit):
    raise RuntimeError("Saved audit table differs")
replay_probe, replay_records = analyze_probe(SOURCE / ARCHIVE_PATH, SOURCE)
if replay_probe != json.loads((PROBE / "summary.json").read_text()):
    raise RuntimeError("Saved reflected-box summary differs from replay")
for name, value in replay_records.items():
    if json.loads(gzip.decompress((PROBE / (name + ".json.gz")).read_bytes())) != value:
        raise RuntimeError("Saved probe record differs: " + name)
probe_members = ["summary.json", *[name + ".json.gz" for name in replay_records]]
saved_inventory = json.loads((PROBE / "inventory.json").read_text())
actual_inventory = {name: {"bytes": (PROBE / name).stat().st_size,
                          "sha256": hashlib.sha256((PROBE / name).read_bytes()).hexdigest()}
                    for name in probe_members}
if saved_inventory != actual_inventory:
    raise RuntimeError("Saved probe inventory differs")
payload = {}
for folder, names in ((AUDIT, ["summary.json", "cases.json", "results.md"]),
                      (PROBE, [*probe_members, "inventory.json"])):
    for name in names:
        payload[folder.name + "/" + name] = (folder / name).read_bytes()
manifest = {"source_commit": SOURCE_COMMIT, "drivers_sha256": DRIVER_HASHES,
            "verification": "all saved analyses exactly replayed", "fresh_inference": False,
            "training": False, "development_only": True,
            "files": {name: {"bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()}
                      for name, data in payload.items()}}
payload["manifest.json"] = (json.dumps(manifest, indent=2) + "\n").encode()
RESULT = WORK / "baomai-detection-audit-r62-results.zip"
with ZipFile(RESULT, "w", ZIP_DEFLATED) as archive:
    for name, data in sorted(payload.items()):
        entry = ZipInfo(name, date_time=(2026, 1, 1, 0, 0, 0))
        entry.compress_type = ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, data)
with ZipFile(RESULT) as archive:
    if archive.testzip() is not None or any(archive.read(name) != data for name, data in payload.items()):
        raise RuntimeError("Saved result ZIP differs")
print("All saved results verified.")
print("Result:", RESULT.name, "bytes:", RESULT.stat().st_size)
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))


All saved results verified.
Result: baomai-detection-audit-r62-results.zip bytes: 701582


/workspace/scratch/f5388ec04e55/r62-notebook-run/work/baomai-detection-audit-r62-results.zip

## Continue the model work

Complete the already prepared Quality Focal Loss versus native-training
comparison first, then the controlled training-label comparison. Both
require GPU training and have their own pinned notebooks. This diagnostic
does not supply those still-pending model results.

For any future genuinely unseen-group evaluation, trace initialization
history: holding a photograph out of fine-tuning is insufficient if the
starting checkpoint was already trained on it.
